# Stage D4 - backward leave-one-out

**Question: is each component *necessary*, given all the others are present?**

Generated in code rather than hand-written, so each arm provably differs from full
BaCP in exactly one lambda.

**The remaining weights are renormalised to 1/3.** Zeroing one of four 0.25 weights
and leaving the rest - what the submitted paper did - removes a term *and*
downweights the whole objective by 25%: two interventions under one label.

### Reading forward against backward

Forward measures **sufficiency** (effect at the all-others-off corner); backward
measures **necessity** (effect at the all-others-on corner). They agree only under
strict additivity, and their gap estimates the aggregate interaction.

| Forward | Backward | Reading |
|---|---|---|
| large | large | genuinely additive |
| large | ~0 | **redundant** - something else already does its job |
| ~0 | large | **synergistic** - only pays off with the rest present |
| ~0 | ~0 | inert; cut it |

There is also a hard reason both are needed: a pure forward ladder has a
lower-triangular design matrix, so with *p* rungs and *p* main effects it has
**zero residual degrees of freedom** and cannot estimate any interaction even in
principle.

Report the gap as its own column - it is the highest-information-per-run quantity
in the design.

> Run only on components that survived the forward pass.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))          # so ladder_nb is importable
import ladder_nb as nb
info = nb.setup()


## Configure

`TIER` 1 is the spine (5 seeds). Cells per GPU and dataloader workers are constants at the top of `pool.py`.


In [ ]:
TIER    = 1
GPUS    = info['gpus'] or 1
SEEDS   = None          # None = every seed the tier schedules

RUNGS = ['D4-noPrC', 'D4-noFiC', 'D4-noSnC', 'D4-noCE']

import manifest as M
grid = [c for c in M.cells(TIER, rungs=RUNGS)]
print(f'{len(grid)} cell(s) planned over {len(set(c["rung"] for c in grid))} rung(s)')
for r in RUNGS:
    n = sum(1 for c in grid if c['rung'] == r)
    print(f'  {r:10s} {n} seed(s)' if n else f'  {r:10s} -- NOT IN TIER {TIER}')


## Run

Idempotent - a cell is complete iff a record carrying its key exists, so re-running skips what is done. Dense cells run first as a hard barrier. Safe to interrupt; you lose at most the cells in flight.


In [ ]:
summary = nb.run_stage(RUNGS, tier=TIER, gpus=GPUS, seeds=SEEDS)
print(summary['ok'], 'ok,', summary['failed'], 'failed,', summary['skipped'], 'skipped')


## Progress and health

The `nan` column is the one to read first. A diverged run sits at exactly 10.0969% (chance on CIFAR-10) for the rest of training and every delta computed from it is meaningless.


In [ ]:
nb.progress(TIER)


## Watch a single cell

Use this when something looks wrong - it streams one line per epoch so you can see *where* a run breaks rather than only that it did.


In [ ]:
cell = nb.attach(nb.pick('D4-noPrC', seed=1, tier=TIER))
nb.show(cell)
# hist, first_nan = nb.watch(cell, gpu=0)
# nb.plot(hist, first_nan, cell['key'])


## Table and gates


In [ ]:
out = nb.report(TIER)
